# Лабораторна робота № 4. ETL-конвеєр для обробки даних

Предметна область – роздрібна торгівля (проєкт «Retail Analytics»), набір даних Online Retail II
(UCI Machine Learning Repository). Конвеєр завантажує сирі продажі (Bronze), очищує їх (Silver),
будує виміри (Gold) і виконує вісім аналітичних завдань на PySpark та Spark SQL.

## 1. Налаштування середовища

Перед запуском у Google Colab файли `online_retail_2009-2010.csv` і `online_retail_2010-2011.csv`
потрібно завантажити на Google Drive у папку `MyDrive/retail-data/`.

In [ ]:
!pip install pyspark

In [ ]:
import sys

from pyspark import SparkConf
from pyspark.sql import SparkSession, Window
import pyspark.sql.types as T
import pyspark.sql.functions as F

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/retail-data/'
else:
    DATA_DIR = 'data/'
RESULTS_DIR = DATA_DIR + 'retail-results/'

# master('local[*]') - використовує всі доступні ядра для обчислень
spark_session = (SparkSession.builder
                 .master('local[*]')
                 .appName('retail etl')
                 .config(conf=SparkConf())
                 .getOrCreate())
print('Spark', spark_session.version)

## 2. Допоміжні функції ETL: E – Extract і L – Load

In [ ]:
def read_data(spark_session, file_name: str, schema: T.StructType):
    """Читає CSV-файл Online Retail II із заданою схемою."""
    return spark_session.read.csv(DATA_DIR + file_name,
                                  header=True,
                                  schema=schema,
                                  timestampFormat='yyyy-MM-dd HH:mm:ss',
                                  mode='FAILFAST')     # помилка, якщо рядок не відповідає схемі


def write_data(df, task_name: str):
    """Записує результат завдання в один CSV-файл у каталозі результатів."""
    output_dir = RESULTS_DIR + task_name
    print(f'Запис результату у: {output_dir}')
    df.coalesce(1).write.csv(output_dir, header=True, mode='overwrite')


def write_layer(df, layer_name: str):
    """Записує шар сховища (Silver або Gold) у форматі Parquet."""
    output_dir = DATA_DIR + 'lakehouse/' + layer_name
    print(f'Запис шару у: {output_dir}')
    df.write.parquet(output_dir, mode='overwrite')

In [ ]:
# Тестовий DataFrame: три рядки чеків
test_df = spark_session.createDataFrame(
    [('536365', '85123A', 'WHITE HANGING HEART T-LIGHT HOLDER', 6, 2.55),
     ('536365', '71053', 'WHITE METAL LANTERN', 6, 3.39),
     ('C536379', 'D', 'Discount', -1, 27.50)],
    schema='invoice string, stock_code string, description string, quantity int, price double')

test_df.withColumn('revenue', F.round(F.col('quantity') * F.col('price'), 2)).show(truncate=False)